In [1]:
import pandas as pd
import time
import os
import urllib.request
import gzip
import shutil
import json
from collections import defaultdict

In [2]:
url = "https://snap.stanford.edu/data/sx-mathoverflow.txt.gz"
file_gz = "sx-mathoverflow.txt.gz"
file_txt = "sx-mathoverflow.txt"

# Download
if not os.path.exists(file_gz):
    print("Downloading dataset...")
    urllib.request.urlretrieve(url, file_gz)
    print("Download complete.")

# Extract
if not os.path.exists(file_txt):
    print("Extracting dataset...")
    with gzip.open(file_gz, "rb") as f_in:
        with open(file_txt, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
    print("Extraction complete.")

print("Dataset ready.")

Dataset ready.


In [3]:
df_txt = pd.read_csv(file_txt, sep=r"\s+", header=None, names=["SRC", "TGT", "Unix"])
df_txt.to_csv("sx-mathoverflow.csv", index=False)

print("Rows:", len(df_txt))
df_txt.head()

Rows: 506550


,SRC,TGT,Unix
0,1,4,1254192988
1,3,4,1254194656
2,1,2,1254202612
3,25,1,1254232804
4,14,16,1254263166


In [4]:
def load_temporal_file(file_path):
    """
    Load temporal network data from CSV or JSON.
    Required logical columns: SRC, TGT, Unix
    """

    if file_path.endswith(".csv"):
        df = pd.read_csv(file_path)

    elif file_path.endswith(".json"):
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        # Case 1: list of records
        if isinstance(data, list):
            df = pd.DataFrame(data)

        # Case 2: dict with edge list
        elif isinstance(data, dict):
            if "edges" in data:
                df = pd.DataFrame(data["edges"])
            else:
                df = pd.DataFrame(data)
        else:
            raise ValueError("Unsupported JSON structure.")

    else:
        raise ValueError("Unsupported file format. Use CSV or JSON.")

    # Normalize column names
    col_map = {}
    for col in df.columns:
        c = str(col).strip().lower()
        if c in ["src", "source", "from"]:
            col_map[col] = "SRC"
        elif c in ["tgt", "target", "to"]:
            col_map[col] = "TGT"
        elif c in ["unix", "time", "timestamp", "t"]:
            col_map[col] = "Unix"

    df = df.rename(columns=col_map)

    required = {"SRC", "TGT", "Unix"}
    if not required.issubset(df.columns):
        raise ValueError(
            f"Missing required columns. Found columns: {df.columns.tolist()}. "
            f"Required columns: SRC, TGT, Unix"
        )

    return df[["SRC", "TGT", "Unix"]]

In [5]:
df_raw = load_temporal_file("sx-mathoverflow.csv")
df_raw.head()

,SRC,TGT,Unix
0,1,4,1254192988
1,3,4,1254194656
2,1,2,1254202612
3,25,1,1254232804
4,14,16,1254263166


In [6]:
print("Number of rows:", len(df_raw))
print("Number of unique nodes:", len(set(df_raw["SRC"]).union(set(df_raw["TGT"]))))
df_raw.describe(include="all")

Number of rows: 506550
Number of unique nodes: 24818


,SRC,TGT,Unix
count,506550.000000,506550.000000,5.065500e+05
mean,12483.063415,15763.243717,1.347550e+09
std,15739.104958,18411.324079,5.872353e+07
min,1.000000,1.000000,1.254193e+09
25%,2051.000000,2811.000000,1.294387e+09
50%,6360.000000,8628.000000,1.343450e+09
75%,15630.000000,22002.000000,1.396941e+09
max,88580.000000,88580.000000,1.457262e+09


In [7]:
def clean_temporal_data(df):
    start = time.time()
    df = df.copy()

    # Standardize datatypes
    df["SRC"] = df["SRC"].astype(str).str.strip()
    df["TGT"] = df["TGT"].astype(str).str.strip()
    df["Unix"] = pd.to_numeric(df["Unix"], errors="coerce")

    # Remove invalid rows
    before_invalid = len(df)
    df = df.dropna(subset=["SRC", "TGT", "Unix"])
    invalid_removed = before_invalid - len(df)

    df["Unix"] = df["Unix"].astype(int)

    # Remove duplicates
    before_dup = len(df)
    df = df.drop_duplicates(subset=["SRC", "TGT", "Unix"])
    duplicates_removed = before_dup - len(df)

    # Sort in strict temporal order
    df = df.sort_values(by=["Unix", "SRC", "TGT"]).reset_index(drop=True)

    # Add stable edge IDs
    df.insert(0, "edge_id", [f"e{str(i).zfill(7)}" for i in range(1, len(df) + 1)])

    # Add datetime column
    df["datetime"] = pd.to_datetime(df["Unix"], unit="s")

    end = time.time()

    print("Cleaning completed.")
    print("Rows after cleaning:", len(df))
    print("Invalid rows removed:", invalid_removed)
    print("Duplicates removed:", duplicates_removed)
    print("Time taken (seconds):", round(end - start, 3))

    return df

In [8]:
df_clean = clean_temporal_data(df_raw)
df_clean.head()

Cleaning completed.
Rows after cleaning: 506523
Invalid rows removed: 0
Duplicates removed: 27
Time taken (seconds): 0.426


,edge_id,SRC,TGT,Unix,datetime
0,e0000001,1,4,1254192988,2009-09-29 02:56:28
1,e0000002,3,4,1254194656,2009-09-29 03:24:16
2,e0000003,1,2,1254202612,2009-09-29 05:36:52
3,e0000004,3,1,1254206196,2009-09-29 06:36:36
4,e0000005,1,1,1254207602,2009-09-29 07:00:02


In [9]:
edges_sorted = sorted(
    zip(df_clean["Unix"], df_clean["SRC"], df_clean["TGT"], df_clean["edge_id"]),
    key=lambda x: (x[0], x[1], x[2])
)

print("First 5 edges_sorted:")
edges_sorted[:5]

First 5 edges_sorted:


[(1254192988, '1', '4', 'e0000001'),
 (1254194656, '3', '4', 'e0000002'),
 (1254202612, '1', '2', 'e0000003'),
 (1254206196, '3', '1', 'e0000004'),
 (1254207602, '1', '1', 'e0000005')]

In [10]:
adj_time = defaultdict(list)

for t, src, tgt, eid in edges_sorted:
    adj_time[src].append((t, tgt, eid))

adj_time = dict(adj_time)

print("Example adjacency entry:")
first_key = next(iter(adj_time))
print(first_key, "->", adj_time[first_key][:5])

Example adjacency entry:
1 -> [(1254192988, '4', 'e0000001'), (1254202612, '2', 'e0000003'), (1254207602, '1', 'e0000005'), (1254259818, '25', 'e0000008'), (1254271421, '16', 'e0000010')]


In [11]:
print("FINAL SUMMARY")
print("---------------------")
print("Total cleaned edges:", len(df_clean))
print("Total unique nodes:", len(set(df_clean["SRC"]).union(set(df_clean["TGT"]))))
print("Memory usage (MB):", round(df_clean.memory_usage(deep=True).sum() / 1e6, 2))
print("Time range:", df_clean["Unix"].min(), "to", df_clean["Unix"].max())

FINAL SUMMARY
---------------------
Total cleaned edges: 506523
Total unique nodes: 24818
Memory usage (MB): 90.95
Time range: 1254192988 to 1457262355


In [12]:
df_subset = df_clean.iloc[:50000].copy()

print("Subset edges:", len(df_subset))
print("Subset memory (MB):", round(df_subset.memory_usage(deep=True).sum() / 1e6, 2))

Subset edges: 50000
Subset memory (MB): 8.9


In [13]:
start_date = df_clean["datetime"].min()
print("Start date:", start_date)

end_date = start_date + pd.Timedelta(days=365)
df_time_window = df_clean[df_clean["datetime"] <= end_date].copy()

print("Edges in 1-year window:", len(df_time_window))
print("Memory (MB):", round(df_time_window.memory_usage(deep=True).sum() / 1e6, 2))

Start date: 2009-09-29 02:56:28
Edges in 1-year window: 97221
Memory (MB): 18.11


In [14]:
df_tmp = df_clean.copy()

start_date = df_tmp["datetime"].min()
target_edges = 50000
candidates = [30, 60, 90, 120, 150, 180, 240, 300, 365]

best = None
for days in candidates:
    end_date = start_date + pd.Timedelta(days=days)
    df_win = df_tmp[df_tmp["datetime"] <= end_date]
    diff = abs(len(df_win) - target_edges)

    if best is None or diff < best[0]:
        best = (diff, days, len(df_win))

print("Best window:")
print("Days:", best[1], "Edges:", best[2], "Diff from target:", best[0])

Best window:
Days: 180 Edges: 42354 Diff from target: 7646


In [15]:
DAYS = best[1]

end_date = start_date + pd.Timedelta(days=DAYS)
df_demo = df_tmp[df_tmp["datetime"] <= end_date].copy()

print("Demo edges:", len(df_demo))
print("Demo nodes:", len(set(df_demo["SRC"]).union(set(df_demo["TGT"]))))
print("Demo memory (MB):", round(df_demo.memory_usage(deep=True).sum() / 1e6, 2))

Demo edges: 42354
Demo nodes: 2256
Demo memory (MB): 7.88


In [16]:
df_clean.to_csv("mathoverflow_cleaned.csv", index=False)
df_demo.to_csv("mathoverflow_demo_42k.csv", index=False)

print("Saved:")
print("- mathoverflow_cleaned.csv")
print("- mathoverflow_demo_42k.csv")

Saved:
- mathoverflow_cleaned.csv
- mathoverflow_demo_42k.csv
